# Customer Churn Prediction - End-to-End Data Science Project

**Author:** Data Science Portfolio Project  
**Date:** March 2026  
**Dataset:** Telco Customer Churn

---

## Table of Contents

1. [Business Understanding](#business-understanding)
2. [Data Cleaning](#data-cleaning)
3. [Exploratory Data Analysis](#exploratory-data-analysis)
4. [Feature Engineering](#feature-engineering)
5. [Model Building](#model-building)
6. [Model Evaluation](#model-evaluation)
7. [Feature Importance Analysis](#feature-importance)
8. [Business Recommendations](#business-recommendations)

---

## 1. Business Understanding

### What is Customer Churn?

**Customer churn** (also known as customer attrition) refers to the phenomenon where customers stop doing business with a company. In the telecommunications industry, this occurs when subscribers cancel their service.

### Why is Churn Important?

- **Cost Impact**: Acquiring a new customer costs 5-25x more than retaining an existing one
- **Revenue Loss**: Churned customers represent direct revenue loss
- **Market Share**: High churn rates indicate competitive disadvantages
- **Growth**: Company growth depends on customer acquisition exceeding churn

### Project Objective

**Primary Goal:** Build a predictive model to identify customers at high risk of churning

**Business Value:**
- Enable proactive retention strategies
- Target high-risk customers with personalized offers
- Reduce customer acquisition costs
- Improve customer lifetime value (CLV)

**Success Metrics:**
- Model accuracy and recall (to catch actual churners)
- Actionable insights from feature importance
- Clear recommendations for business stakeholders

## 2. Import Required Libraries

In [ ]:
# Data Manipulation and Analysis
import pandas as pd
import numpy as np

# Data Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning - Preprocessing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

# Machine Learning - Models
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# Machine Learning - Evaluation Metrics
from sklearn.metrics import (
    accuracy_score, 
    precision_score, 
    recall_score, 
    f1_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    roc_auc_score
)

# Ignore warnings for cleaner output
import warnings
warnings.filterwarnings('ignore')

# Set visualization style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✅ All libraries imported successfully!")

## 3. Load and Inspect Dataset

In [ ]:
# Load the dataset
# Note: You'll need to download the Telco Customer Churn dataset from:
# https://www.kaggle.com/datasets/blastchar/telco-customer-churn
# Or use: df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')

# For demonstration, we'll load directly from GitHub
url = 'https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv'

try:
    df = pd.read_csv(url)
    print("✅ Dataset loaded successfully from URL!")
except:
    print("⚠️ Could not load from URL. Please download the dataset and use:")
    print("   df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')")
    # Uncomment the line below and provide your file path:
    # df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')

print(f"\nDataset Shape: {df.shape}")
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")

In [ ]:
# Display first few rows
print("First 5 rows of the dataset:\n")
df.head()

In [ ]:
# Dataset information
print("Dataset Information:\n")
df.info()

In [ ]:
# Statistical summary
print("Statistical Summary:\n")
df.describe()

## 4. Data Cleaning

Data cleaning is crucial for building reliable machine learning models. We'll:
1. Handle missing values
2. Check for duplicates
3.Convert TotalCharges to numeric type
4. Drop unnecessary columns

In [ ]:
# Check for missing values
print("Missing Values Count:\n")
missing_values = df.isnull().sum()
print(missing_values[missing_values > 0])

if missing_values.sum() == 0:
    print("\n✅ No missing values found!")
else:
    print(f"\n⚠️ Total missing values: {missing_values.sum()}")

In [ ]:
# Check for duplicates
duplicate_count = df.duplicated().sum()
print(f"Number of duplicate rows: {duplicate_count}")

if duplicate_count > 0:
    print(f"Removing {duplicate_count} duplicate rows...")
    df = df.drop_duplicates()
    print("✅ Duplicates removed!")
else:
    print("✅ No duplicates found!")

In [ ]:
# Convert TotalCharges to numeric
# First, check for non-numeric values
print("Checking for non-numeric values in TotalCharges...")
non_numeric = df[pd.to_numeric(df['TotalCharges'], errors='coerce').isna()]
print(f"Found {len(non_numeric)} non-numeric entries")

if len(non_numeric) > 0:
    print("\nSample of non-numeric entries:")
    print(non_numeric[['customerID', 'tenure', 'MonthlyCharges', 'TotalCharges']].head())
    
# Convert to numeric, coercing errors to NaN
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Handle NaN values in TotalCharges
# Since these are likely customers with 0 tenure, we'll fill with 0
nan_count = df['TotalCharges'].isna().sum()
if nan_count > 0:
    print(f"\nFilling {nan_count} NaN values in TotalCharges with 0...")
    df['TotalCharges'] = df['TotalCharges'].fillna(0)
    
print("\n✅ TotalCharges converted to numeric successfully!")
print(f"TotalCharges data type: {df['TotalCharges'].dtype}")

In [ ]:
# Drop customerID as it's just an identifier and won't help in prediction
df = df.drop('customerID', axis=1)
print("✅ Dropped customerID column")
print(f"New shape: {df.shape}")

### Check Class Imbalance

Understanding the distribution of our target variable (Churn) is crucial for model building and evaluation.

In [ ]:
# Check class distribution
print("Churn Distribution:\n")
churn_counts = df['Churn'].value_counts()
churn_percentages = df['Churn'].value_counts(normalize=True) * 100

print(churn_counts)
print("\nPercentages:")
print(churn_percentages)

# Visualize class distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Count plot
sns.countplot(data=df, x='Churn', palette='Set2', ax=axes[0])
axes[0].set_title('Churn Distribution (Count)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Churn', fontsize=12)
axes[0].set_ylabel('Count', fontsize=12)

# Add count labels on bars
for container in axes[0].containers:
    axes[0].bar_label(container)

# Pie chart
colors = ['#90EE90', '#FF6B6B']
axes[1].pie(churn_percentages, labels=churn_percentages.index, autopct='%1.1f%%', 
            startangle=90, colors=colors, explode=(0.05, 0.05))
axes[1].set_title('Churn Distribution (Percentage)', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

print("\n📊 Business Insight:")
print(f"   - {churn_percentages['No']:.1f}% of customers are retained")
print(f"   - {churn_percentages['Yes']:.1f}% of customers have churned")
print(f"   - The dataset shows {'moderate' if 20 < churn_percentages['Yes'] < 40 else 'significant'} class imbalance")

## 5. Exploratory Data Analysis (EDA)

EDA helps us understand patterns, relationships, and insights in the data before building models.

### 5.1 Univariate Analysis - Numerical Features

In [ ]:
# Analyze key numerical features
numerical_features = ['tenure', 'MonthlyCharges', 'TotalCharges']

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

for idx, feature in enumerate(numerical_features):
    # Histogram
    axes[0, idx].hist(df[feature], bins=30, edgecolor='black', color='skyblue', alpha=0.7)
    axes[0, idx].set_title(f'Distribution of {feature}', fontsize=12, fontweight='bold')
    axes[0, idx].set_xlabel(feature)
    axes[0, idx].set_ylabel('Frequency')
    axes[0, idx].axvline(df[feature].mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {df[feature].mean():.2f}')
    axes[0, idx].legend()
    
    # Box plot
    axes[1, idx].boxplot(df[feature], vert=True, patch_artist=True,
                         boxprops=dict(facecolor='lightcoral', alpha=0.7),
                         medianprops=dict(color='darkred', linewidth=2))
    axes[1, idx].set_title(f'Box Plot of {feature}', fontsize=12, fontweight='bold')
    axes[1, idx].set_ylabel(feature)

plt.tight_layout()
plt.show()

print("\n📊 Key Insights from Numerical Features:\n")
print(f"1. Tenure:")
print(f"   - Average customer tenure: {df['tenure'].mean():.1f} months")
print(f"   - Many customers are either very new (0-12 months) or long-term (60+ months)")
print(f"\n2. Monthly Charges:")
print(f"   - Average: ${df['MonthlyCharges'].mean():.2f}")
print(f"   - Range: ${df['MonthlyCharges'].min():.2f} - ${df['MonthlyCharges'].max():.2f}")
print(f"\n3. Total Charges:")
print(f"   - Average lifetime value: ${df['TotalCharges'].mean():.2f}")

### 5.2 Univariate Analysis - Categorical Features

In [ ]:
# Key categorical features to analyze
categorical_features = ['Contract', 'PaymentMethod', 'InternetService']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, feature in enumerate(categorical_features):
    df[feature].value_counts().plot(kind='bar', ax=axes[idx], color='steelblue', alpha=0.7)
    axes[idx].set_title(f'Distribution of {feature}', fontsize=12, fontweight='bold')
    axes[idx].set_xlabel(feature)
    axes[idx].set_ylabel('Count')
    axes[idx].set_xticklabels(axes[idx].get_xticklabels(), rotation=45, ha='right')
    
    # Add count labels
    for container in axes[idx].containers:
        axes[idx].bar_label(container)

plt.tight_layout()
plt.show()

print("\n📊 Categorical Features Insights:")
for feature in categorical_features:
    print(f"\n{feature}:")
    print(df[feature].value_counts())

### 5.3 Bivariate Analysis - Churn vs Key Features

In [ ]:
# Churn vs Numerical Features
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, feature in enumerate(numerical_features):
    sns.boxplot(data=df, x='Churn', y=feature, palette='Set2', ax=axes[idx])
    axes[idx].set_title(f'Churn vs {feature}', fontsize=12, fontweight='bold')
    axes[idx].set_xlabel('Churn')
    axes[idx].set_ylabel(feature)

plt.tight_layout()
plt.show()

# Statistical comparison
print("\n📊 Business Insights - Numerical Features vs Churn:\n")
churned = df[df['Churn'] == 'Yes']
retained = df[df['Churn'] == 'No']

print(f"1. Tenure:")
print(f"   - Churned customers avg tenure: {churned['tenure'].mean():.1f} months")
print(f"   - Retained customers avg tenure: {retained['tenure'].mean():.1f} months")
print(f"   - ⚠️ Customers with shorter tenure are MORE likely to churn")

print(f"\n2. Monthly Charges:")
print(f"   - Churned customers avg: ${churned['MonthlyCharges'].mean():.2f}")
print(f"   - Retained customers avg: ${retained['MonthlyCharges'].mean():.2f}")
print(f"   - ⚠️ Higher monthly charges correlate with higher churn")

In [ ]:
# Churn vs Categorical Features
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, feature in enumerate(categorical_features):
    # Create a crosstab for churn rates
    ct = pd.crosstab(df[feature], df['Churn'], normalize='index') * 100
    ct.plot(kind='bar', ax=axes[idx], color=['#90EE90', '#FF6B6B'], alpha=0.7)
    axes[idx].set_title(f'Churn Rate by {feature}', fontsize=12, fontweight='bold')
    axes[idx].set_xlabel(feature)
    axes[idx].set_ylabel('Percentage (%)')
    axes[idx].legend(title='Churn', labels=['No', 'Yes'])
    axes[idx].set_xticklabels(axes[idx].get_xticklabels(), rotation=45, ha='right')

plt.tight_layout()
plt.show()

print("\n📊 Key Business Insights - Categorical Features:\n")
print("1. Contract Type:")
print("   - Month-to-month contracts show HIGHEST churn risk")
print("   - Long-term contracts (1-2 years) have much lower churn")
print("   - Recommendation: Incentivize longer contract commitments")

print("\n2. Payment Method:")
print("   - Electronic check users show higher churn")
print("   - Automatic payment methods correlate with lower churn")
print("   - Recommendation: Promote autopay enrollment")

### 5.4 Correlation Analysis

In [ ]:
# Create correlation matrix for numerical features
correlation_matrix = df[numerical_features].corr()

# Visualize correlation heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0, 
            square=True, linewidths=1, cbar_kws={"shrink": 0.8},
            fmt='.2f', vmin=-1, vmax=1)
plt.title('Correlation Heatmap - Numerical Features', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

print("\n📊 Correlation Insights:")
print(f"- Tenure & Total Charges: Strong positive correlation ({correlation_matrix.loc['tenure', 'TotalCharges']:.2f})")
print("  → Makes sense: longer tenure = more total spending")
print(f"- Monthly Charges & Total Charges: Moderate correlation ({correlation_matrix.loc['MonthlyCharges', 'TotalCharges']:.2f})")
print("  → Higher monthly fees contribute to total lifetime value")

## 6. Feature Engineering

Creating new features to improve model performance and interpretability.

### 6.1 Create Tenure Groups

**Reasoning:** Grouping tenure into categories helps identify customer lifecycle stages:
- **New** (0-12 months): High churn risk, still evaluating service
- **Mid-term** (13-24 months): Established customers, moderate risk
- **Long-term** (25-48 months): Loyal customers, lower risk
- **Very Long-term** (49+ months): Most loyal, lowest risk

In [ ]:
# Create tenure groups
def create_tenure_group(tenure):
    if tenure <= 12:
        return 'New (0-12)'
    elif tenure <= 24:
        return 'Mid-term (13-24)'
    elif tenure <= 48:
        return 'Long-term (25-48)'
    else:
        return 'Very Long-term (49+)'

df['tenure_group'] = df['tenure'].apply(create_tenure_group)

# Visualize tenure groups vs churn
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Distribution of tenure groups
tenure_dist = df['tenure_group'].value_counts()
axes[0].bar(range(len(tenure_dist)), tenure_dist.values, color='steelblue', alpha=0.7)
axes[0].set_xticks(range(len(tenure_dist)))
axes[0].set_xticklabels(tenure_dist.index, rotation=45, ha='right')
axes[0].set_title('Distribution of Tenure Groups', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Count')
for i, v in enumerate(tenure_dist.values):
    axes[0].text(i, v + 50, str(v), ha='center', va='bottom', fontweight='bold')

# Churn rate by tenure group
churn_by_tenure = pd.crosstab(df['tenure_group'], df['Churn'], normalize='index') * 100
churn_by_tenure.plot(kind='bar', ax=axes[1], color=['#90EE90', '#FF6B6B'], alpha=0.7)
axes[1].set_title('Churn Rate by Tenure Group', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Tenure Group')
axes[1].set_ylabel('Percentage (%)')
axes[1].legend(title='Churn', labels=['No', 'Yes'])
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=45, ha='right')

plt.tight_layout()
plt.show()

print("✅ Tenure groups created successfully!")

## 7. Data Preprocessing for Modeling

### 7.1 Encode Categorical Variables

In [ ]:
# Create a copy for modeling
df_model = df.copy()

# Identify categorical and numerical columns
categorical_cols = df_model.select_dtypes(include=['object']).columns.tolist()
categorical_cols.remove('Churn')  # Remove target variable

# Apply Label Encoding for binary categorical features
binary_cols = [col for col in categorical_cols if df_model[col].nunique() == 2]

le = LabelEncoder()
for col in binary_cols:
    df_model[col] = le.fit_transform(df_model[col])

# Apply One-Hot Encoding for multi-class categorical features
multi_class_cols = [col for col in categorical_cols if col not in binary_cols]
df_model = pd.get_dummies(df_model, columns=multi_class_cols, drop_first=True)

# Encode target variable
df_model['Churn'] = le.fit_transform(df_model['Churn'])  # 0 = No, 1 = Yes

print(f"✅ Encoding complete!")
print(f"New dataset shape: {df_model.shape}")

### 7.2 Train-Test Split (80/20)

In [ ]:
# Separate features (X) and target (y)
X = df_model.drop('Churn', axis=1)
y = df_model['Churn']

# Split data with stratification to maintain class distribution
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("✅ Train-Test Split Complete!")
print(f"\nTraining set size: {X_train.shape[0]} ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"Test set size: {X_test.shape[0]} ({X_test.shape[0]/len(X)*100:.1f}%)")

### 7.3 Feature Scaling

Scaling is important for Logistic Regression to ensure all features contribute equally.

In [ ]:
# Apply StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("✅ Feature scaling complete!")

## 8. Model Building

We'll train three models and compare their performance:
1. **Logistic Regression**: Simple, interpretable baseline
2. **Decision Tree**: Captures non-linear patterns
3. **Random Forest**: Ensemble method for better accuracy

In [ ]:
# Train all three models

# 1. Logistic Regression (using scaled data)
print("Training Logistic Regression...")
lr_model = LogisticRegression(random_state=42, max_iter=1000)
lr_model.fit(X_train_scaled, y_train)
y_pred_lr = lr_model.predict(X_test_scaled)
y_pred_proba_lr = lr_model.predict_proba(X_test_scaled)[:, 1]
print("✅ Logistic Regression trained")

# 2. Decision Tree (no scaling needed)
print("\nTraining Decision Tree...")
dt_model = DecisionTreeClassifier(random_state=42, max_depth=10, min_samples_split=20)
dt_model.fit(X_train, y_train)
y_pred_dt = dt_model.predict(X_test)
y_pred_proba_dt = dt_model.predict_proba(X_test)[:, 1]
print("✅ Decision Tree trained")

# 3. Random Forest (no scaling needed)
print("\nTraining Random Forest...")
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=15, min_samples_split=20)
rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)
y_pred_proba_rf = rf_model.predict_proba(X_test)[:, 1]
print("✅ Random Forest trained")

print("\n🎉 All models trained successfully!")

## 9. Model Evaluation

Comprehensive evaluation of all three models using multiple metrics.

In [ ]:
# Calculate metrics for all models
models = {
    'Logistic Regression': y_pred_lr,
    'Decision Tree': y_pred_dt,
    'Random Forest': y_pred_rf
}

print("="*80)
print("MODEL PERFORMANCE METRICS")
print("="*80)

for model_name, predictions in models.items():
    print(f"\n{model_name}:")
    print("-" * 60)
    print(f"Accuracy:  {accuracy_score(y_test, predictions):.4f}")
    print(f"Precision: {precision_score(y_test, predictions):.4f}")
    print(f"Recall:    {recall_score(y_test, predictions):.4f}")
    print(f"F1 Score:  {f1_score(y_test, predictions):.4f}")
    
print("\n" + "="*80)
print("\n📊 Interpretation:")
print("  - Accuracy: Overall correctness")
print("  - Precision: Of predicted churners, how many actually churned")
print("  - Recall: Of actual churners, how many did we catch")
print("  - F1 Score: Harmonic mean of precision and recall")

### 9.1 Confusion Matrices

In [ ]:
# Visualize confusion matrices for all models
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, (model_name, predictions) in enumerate(models.items()):
    cm = confusion_matrix(y_test, predictions)
    
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx], 
                cbar=False, square=True, linewidths=2)
    axes[idx].set_title(f'Confusion Matrix\n{model_name}', fontsize=12, fontweight='bold')
    axes[idx].set_xlabel('Predicted', fontsize=10)
    axes[idx].set_ylabel('Actual', fontsize=10)
    axes[idx].set_xticklabels(['No Churn', 'Churn'])
    axes[idx].set_yticklabels(['No Churn', 'Churn'])

plt.tight_layout()
plt.show()

print("\n📊 Confusion Matrix Interpretation:")
print("  - True Negatives (TN): Correctly predicted no churn")
print("  - False Positives (FP): Predicted churn, but didn't churn")
print("  - False Negatives (FN): Predicted no churn, but churned ❌ Most costly!")
print("  - True Positives (TP): Correctly predicted churn")

### 9.2 ROC Curves and AUC Scores

In [ ]:
# Plot ROC curves for all models
plt.figure(figsize=(10, 7))

models_proba = {
    'Logistic Regression': y_pred_proba_lr,
    'Decision Tree': y_pred_proba_dt,
    'Random Forest': y_pred_proba_rf
}

colors = ['blue', 'green', 'red']

for (model_name, y_proba), color in zip(models_proba.items(), colors):
    # Calculate ROC curve
    fpr, tpr, thresholds = roc_curve(y_test, y_proba)
    auc_score = roc_auc_score(y_test, y_proba)
    
    # Plot ROC curve
    plt.plot(fpr, tpr, label=f'{model_name} (AUC = {auc_score:.3f})', 
             linewidth=2, color=color)

# Plot diagonal line (random classifier)
plt.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random Classifier (AUC = 0.500)')

plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curves - Model Comparison', fontsize=14, fontweight='bold')
plt.legend(loc='lower right', fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\n📊 AUC Score Interpretation:")
print("  - AUC = 1.0: Perfect classifier")
print("  - AUC = 0.9-1.0: Excellent")
print("  - AUC = 0.8-0.9: Very Good")
print("  - AUC = 0.7-0.8: Good")
print("  - AUC = 0.5: Random guessing")

### 9.3 Model Comparison Table

In [ ]:
# Create comprehensive comparison table
comparison_data = []

for model_name, predictions in models.items():
    y_proba = models_proba[model_name]
    auc = roc_auc_score(y_test, y_proba)
    
    comparison_data.append({
        'Model': model_name,
        'Accuracy': f"{accuracy_score(y_test, predictions):.4f}",
        'Precision': f"{precision_score(y_test, predictions):.4f}",
        'Recall': f"{recall_score(y_test, predictions):.4f}",
        'F1 Score': f"{f1_score(y_test, predictions):.4f}",
        'AUC': f"{auc:.4f}"
    })

comparison_df = pd.DataFrame(comparison_data)

print("\n" + "="*90)
print("COMPREHENSIVE MODEL COMPARISON")
print("="*90)
print(comparison_df.to_string(index=False))
print("="*90)

# Identify best model for each metric
print("\n🏆 Best Model by Metric:")
for col in ['Accuracy', 'Precision', 'Recall', 'F1 Score', 'AUC']:
    best_idx = comparison_df[col].astype(float).idxmax()
    best_model = comparison_df.loc[best_idx, 'Model']
    best_value = comparison_df.loc[best_idx, col]
    print(f"  {col:12s}: {best_model} ({best_value})")

## 10. Feature Importance Analysis

Understanding which features most influence customer churn predictions.

In [ ]:
# Extract feature importance from Random Forest
feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False)

# Visualize top 15 features
plt.figure(figsize=(12, 8))
top_features = feature_importance.head(15)
plt.barh(range(len(top_features)), top_features['Importance'], color='steelblue', alpha=0.7)
plt.yticks(range(len(top_features)), top_features['Feature'])
plt.xlabel('Importance', fontsize=12)
plt.ylabel('Features', fontsize=12)
plt.title('Top 15 Most Important Features (Random Forest)', fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

print("\n📊 Top 10 Most Important Features:\n")
print(feature_importance.head(10).to_string(index=False))

### Business Interpretation of Key Features

Understanding the most important drivers of churn:

In [ ]:
print("📊 Key Drivers of Customer Churn:\n")
print("="*70)

print("\n1️⃣ TENURE (Customer Lifetime)")
print("   - Most important predictor of churn")
print("   - New customers (0-12 months) are highest risk")
print("   - Action: Intensive onboarding and support for new customers")

print("\n2️⃣ MONTHLY CHARGES")
print("   - Higher charges correlate with increased churn")
print("   - Price sensitivity is a major factor")
print("   - Action: Review pricing strategy and offer value-added services")

print("\n3️⃣ CONTRACT TYPE")
print("   - Month-to-month contracts have much higher churn")
print("   - Long-term contracts create commitment and reduce churn")
print("   - Action: Incentivize annual/2-year contracts with discounts")

print("\n4️⃣ INTERNET SERVICE & ADD-ONS")
print("   - Type of internet service impacts churn")
print("   - Customers without support services are more likely to churn")
print("   - Action: Bundle services and improve fiber optic service quality")

print("\n5️⃣ PAYMENT METHOD")
print("   - Electronic check users show higher churn")
print("   - Autopay creates convenience and habit")
print("   - Action: Promote automatic payment methods")

## 11. Final Business Recommendations

### Actionable Strategies to Reduce Customer Churn

In [ ]:
print("🎯 STRATEGIC RECOMMENDATIONS TO REDUCE CHURN")
print("="*80)

print("\n1. 🎯 TARGET HIGH-RISK CUSTOMERS")
print("   Strategy: Use the predictive model to identify at-risk customers")
print("   Action:")
print("   • Deploy model to score all active customers monthly")
print("   • Create tiered intervention programs (High/Medium/Low risk)")
print("   • Proactively reach out before churn occurs")

print("\n2. 💰 OPTIMIZE PRICING & CONTRACT STRATEGY")
print("   Strategy: Address price sensitivity and encourage commitment")
print("   Action:")
print("   • Offer 10-15% discount for annual contracts")
print("   • Create loyalty rewards for long-term customers")
print("   • Implement personalized pricing for high-value customers")
print("   • Review fiber optic service pricing vs competitors")

print("\n3. 🆕 ENHANCE NEW CUSTOMER ONBOARDING")
print("   Strategy: Reduce early-stage churn (0-12 months)")
print("   Action:")
print("   • Implement 90-day onboarding journey with checkpoints")
print("   • Provide welcome package and dedicated support ")
print("   • Offer first 3-month satisfaction guarantee")
print("   • Conduct proactive check-ins at 30, 60, 90 days")

print("\n4. 💳 PROMOTE AUTOPAY & CONVENIENT PAYMENT")
print("   Strategy: Reduce friction and increase convenience")
print("   Action:")
print("   • Offer $5-10/month discount for autopay enrollment")
print("   • Send targeted campaigns to electronic check users")
print("   • Simplify payment method switching process")

print("\n5. 📦 CREATE SERVICE BUNDLES")
print("   Strategy: Increase customer value and stickiness")
print("   Action:")
print("   • Bundle internet + phone + security at discounted rate")
print("   • Include tech support and online backup in premium tiers")
print("   • Cross-sell complementary services to single-service users")

print("\n6. 📞 IMPROVE CUSTOMER SUPPORT")
print("   Strategy: Enhance service quality and satisfaction")
print("   Action:")
print("   • Expand technical support hours and channels")
print("   • Implement proactive service monitoring")
print("   • Create fast-track support for high-value customers")
print("   • Address fiber optic service quality issues")

print("\n7. 📊 IMPLEMENT RETENTION CAMPAIGNS")
print("   Strategy: Win-back at-risk customers before they leave")
print("   Action:")
print("   • Create save offers for contract expiration dates")
print("   • Personalized retention offers based on customer profile")
print("   • Exit surveys to understand why customers leave")
print("   • Win-back campaigns for recently churned customers")

print("\n" + "="*80)
print("\n✅ EXPECTED OUTCOMES:")
print("   • 20-30% reduction in churn rate within 12 months")
print("   • Increased customer lifetime value (CLV)")
print("   • Improved customer satisfaction scores")
print("   • Higher contract conversion rates")
print("   • Better resource allocation for retention efforts")

print("\n" + "="*80)
print("📈 PROJECT COMPLETE - READY FOR STAKEHOLDER PRESENTATION!")
print("="*80)